# SenseForge — Independent WSD Benchmark

## Objective
Evaluate the generalization of three WSD architectures:

1. Frozen XLM-R + Logistic Regression
2. Frozen XLM-R + Residual MLP
3. XLM-R + LoRA fine-tuning

## Evaluation principles
- No training on benchmark examples
- No hyperparameter tuning using test results
- Explicit BabelNet-to-Contexta sense verification
- Sentence-level leakage prevention
- Consistent lemma-restricted inference
- Per-class accuracy, precision, recall and Macro-F1

## Benchmark tracks
1. Existing V2 held-out test — previously evaluated
2. Fresh external natural-context benchmark — pending collection
3. Hindi-English code-mixed robustness — separate evaluation

In [3]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

ART = ROOT / "artifacts"
V2 = ART / "wsd_v2"
SUP = ART / "wsd_v2_supervised"
ADV = ART / "wsd_v2_advanced"
LORA = ART / "wsd_v2_lora"
OUT = ART / "wsd_independent_benchmark"

OUT.mkdir(parents=True, exist_ok=True)

required = {
    "V2 test": V2 / "test.jsonl",
    "LogReg": SUP / "best_supervised_model.joblib",
    "Residual MLP": ADV / "advanced_residual_mlp.pt",
    "MLP scaler": ADV / "scaler.joblib",
    "LoRA adapter": LORA / "best_lora_adapter",
    "LoRA head": LORA / "best_classifier_head.pt"
}

for name, path in required.items():
    exists = path.exists()
    print(f"{name:20} {'FOUND' if exists else 'MISSING'}")
    assert exists, f"Missing artifact: {path}"

print("\nArtifact verification passed.")

V2 test              FOUND
LogReg               FOUND
Residual MLP         FOUND
MLP scaler           FOUND
LoRA adapter         FOUND
LoRA head            FOUND

Artifact verification passed.


In [5]:
from collections import Counter

def load_jsonl(path):
    with path.open(encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

test_rows = load_jsonl(V2 / "test.jsonl")

SENSES = [
    "bank.1", "bank.2",
    "interest.1", "interest.2",
    "plant.1", "plant.2",
    "point.1", "point.2",
    "right.1", "right.2",
    "table.1", "table.2"
]

assert len(test_rows) == 62

for row in test_rows:
    assert row["sense_id"] in SENSES
    assert row["sense_id"].startswith(
        row["target_word"] + "."
    )

print("Existing test records:", len(test_rows))
print("Sense distribution:")

for sense, count in sorted(
    Counter(r["sense_id"] for r in test_rows).items()
):
    print(f"{sense:15} {count:3}")

Existing test records: 62
Sense distribution:
bank.1            3
bank.2            3
interest.1        3
interest.2        4
plant.1           3
plant.2          14
point.1          10
point.2           7
right.1           2
right.2           1
table.1           3
table.2           9


In [7]:
protocol = {
    "name": "SenseForge Independent WSD Benchmark",
    "version": "1.0",
    "target_words": [
        "bank", "interest", "plant",
        "point", "right", "table"
    ],
    "sense_labels": SENSES,
    "models": [
        "xlmr_logistic_regression",
        "xlmr_residual_mlp",
        "xlmr_lora"
    ],
    "evaluation_tracks": {
        "existing_v2_test": {
            "records": len(test_rows),
            "status": "previously_evaluated",
            "eligible_for_model_selection": False
        },
        "external_gold": {
            "status": "not_collected",
            "eligible_for_model_selection": False
        },
        "code_mixed": {
            "status": "not_collected",
            "eligible_for_model_selection": False
        }
    },
    "metrics": [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "per_sense_f1"
    ],
    "restrictions": {
        "no_test_training": True,
        "no_test_hyperparameter_tuning": True,
        "lemma_restricted_inference": True,
        "sentence_group_leakage_check": True
    }
}

path = OUT / "benchmark_protocol.json"

path.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print("Benchmark protocol saved:", path)

Benchmark protocol saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/wsd_independent_benchmark/benchmark_protocol.json


In [9]:
XLWSD = ROOT / "datasets" / "xl-wsd"

print("XL-WSD root:", XLWSD)

for path in sorted(XLWSD.iterdir()):
    print(
        "[DIR]" if path.is_dir() else "[FILE]",
        path.name
    )

print("\nPotential evaluation files:")

matches = []

for path in XLWSD.rglob("*"):
    if not path.is_file():
        continue

    name = str(path.relative_to(XLWSD)).lower()

    if any(key in name for key in [
        "evaluation", "test", "dev", "semeval"
    ]):
        matches.append(path)

for path in matches[:60]:
    print(path.relative_to(XLWSD))

print("\nCandidate files found:", len(matches))

XL-WSD root: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/datasets/xl-wsd
[FILE] .DS_Store
[FILE] LICENSE
[FILE] README.md
[FILE] evaluate_answers.py
[DIR] evaluation_datasets
[DIR] inventories
[DIR] training_datasets

Potential evaluation files:
evaluation_datasets/dev-zh/dev-zh.gold.key.txt
evaluation_datasets/dev-zh/dev-zh.data.xml
evaluation_datasets/test-da/test-da.data.xml
evaluation_datasets/test-da/test-da.gold.key.txt
evaluation_datasets/dev-ko/dev-ko.gold.key.txt
evaluation_datasets/dev-ko/dev-ko.data.xml
evaluation_datasets/test-sl/test-sl.gold.key.txt
evaluation_datasets/test-sl/test-sl.data.xml
evaluation_datasets/dev-de/dev-de.data.xml
evaluation_datasets/dev-de/dev-de.gold.key.txt
evaluation_datasets/dev-bg/dev-bg.data.xml
evaluation_datasets/dev-bg/dev-bg.gold.key.txt
evaluation_datasets/dev-nl/dev-nl.data.xml
evaluation_datasets/dev-nl/dev-nl.gold.key.txt
evaluation_datasets/dev-hu/dev-hu.data.xml
evaluation_datasets/dev-hu/dev-hu.gold.key.txt
evaluatio

In [11]:
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict

XLWSD = ROOT / "datasets" / "xl-wsd"
EVAL = XLWSD / "evaluation_datasets"

mapping = json.loads(
    (ROOT / "configs/babelnet_to_contexta.json").read_text(
        encoding="utf-8"
    )
)

WORDS = {
    "bank", "interest", "plant",
    "point", "right", "table"
}

def audit_external_dataset(name):
    folder = EVAL / name
    xml_path = folder / f"{name}.data.xml"
    key_path = folder / f"{name}.gold.key.txt"

    instances = {}

    for _, elem in ET.iterparse(
        xml_path, events=("end",)
    ):
        if elem.tag == "instance":
            lemma = elem.get("lemma", "").lower()

            if lemma in WORDS:
                instances[elem.get("id")] = lemma

        elem.clear()

    counts = Counter()
    mapped_counts = Counter()
    skipped = Counter()
    unique_senses = defaultdict(set)

    with key_path.open(encoding="utf-8") as f:
        for line in f:
            parts = line.split()

            if len(parts) < 2:
                continue

            instance_id = parts[0]

            if instance_id not in instances:
                continue

            word = instances[instance_id]
            labels = parts[1:]

            counts[word] += 1
            unique_senses[word].update(labels)

            mapped = [
                mapping.get(s)
                for s in labels
            ]

            if any(s is None for s in mapped):
                skipped["unmapped_or_partial"] += 1
                continue

            if len(set(mapped)) != 1:
                skipped["conflicting"] += 1
                continue

            sense = mapped[0]

            if not sense.startswith(word + "."):
                skipped["lemma_mismatch"] += 1
                continue

            mapped_counts[sense] += 1

    print(f"\n=== {name} ===")
    print("Target occurrences:", sum(counts.values()))
    print("Mapped occurrences:", sum(mapped_counts.values()))
    print("Skipped:", dict(skipped))

    print("\nWord coverage:")
    for word in sorted(WORDS):
        print(
            f"{word:12} "
            f"total={counts[word]:4} "
            f"unique_senses={len(unique_senses[word]):3}"
        )

    print("\nMapped sense distribution:")
    for sense, n in sorted(mapped_counts.items()):
        print(f"{sense:15} {n:4}")


audit_external_dataset("dev-en")
audit_external_dataset("test-en-coarse")


=== dev-en ===
Target occurrences: 0
Mapped occurrences: 0
Skipped: {}

Word coverage:
bank         total=   0 unique_senses=  0
interest     total=   0 unique_senses=  0
plant        total=   0 unique_senses=  0
point        total=   0 unique_senses=  0
right        total=   0 unique_senses=  0
table        total=   0 unique_senses=  0

Mapped sense distribution:

=== test-en-coarse ===
Target occurrences: 6
Mapped occurrences: 0
Skipped: {'unmapped_or_partial': 6}

Word coverage:
bank         total=   0 unique_senses=  0
interest     total=   0 unique_senses=  0
plant        total=   1 unique_senses=  5
point        total=   1 unique_senses=  5
right        total=   3 unique_senses=  5
table        total=   1 unique_senses=  4

Mapped sense distribution:


In [13]:
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

EVAL = ROOT / "datasets/xl-wsd/evaluation_datasets"

for name in ["dev-en", "test-en-coarse"]:
    path = EVAL / name / f"{name}.data.xml"

    print(f"\n=== {name} ===")

    root = ET.parse(path).getroot()

    print("Root tag:", root.tag)
    print("Root attributes:", root.attrib)

    instances = root.findall(".//instance")

    print("Total XML instances:", len(instances))

    for elem in instances[:10]:
        print({
            "id": elem.get("id"),
            "lemma": elem.get("lemma"),
            "pos": elem.get("pos"),
            "surface": (elem.text or "").strip()
        })

    lemmas = Counter(
        (elem.get("lemma") or "").lower()
        for elem in instances
    )

    print("\nTarget lemma matches:")
    for word in [
        "bank", "interest", "plant",
        "point", "right", "table"
    ]:
        print(word, lemmas[word])


=== dev-en ===
Root tag: corpus
Root attributes: {'lang': 'en', 'source': 'semeval2007'}
Total XML instances: 455
{'id': 'd000.s000.t000', 'lemma': 'refer', 'pos': 'VERB', 'surface': 'referred'}
{'id': 'd000.s000.t001', 'lemma': 'research', 'pos': 'NOUN', 'surface': 'research'}
{'id': 'd000.s000.t002', 'lemma': 'report', 'pos': 'VERB', 'surface': 'reported'}
{'id': 'd000.s001.t000', 'lemma': 'comment', 'pos': 'NOUN', 'surface': 'comments'}
{'id': 'd000.s001.t001', 'lemma': 'imply', 'pos': 'VERB', 'surface': 'implied'}
{'id': 'd000.s001.t002', 'lemma': 'discover', 'pos': 'VERB', 'surface': 'discovered'}
{'id': 'd000.s001.t003', 'lemma': 'cause', 'pos': 'NOUN', 'surface': 'cause'}
{'id': 'd000.s001.t004', 'lemma': 'find', 'pos': 'VERB', 'surface': 'found'}
{'id': 'd000.s002.t000', 'lemma': 'make', 'pos': 'VERB', 'surface': 'made'}
{'id': 'd000.s002.t001', 'lemma': 'statement', 'pos': 'NOUN', 'surface': 'statement'}

Target lemma matches:
bank 0
interest 0
plant 0
point 0
right 0
table 0